This is the tutorial for the evaluation of CPKS alchemical derivatives using PySCF (version 1.7.6)

In [1]:
from pyscf import gto,scf
import numpy as np
import pyscf
pyscf.__version__

/usr/local/lib/python3.9/site-packages/pyscf/lib/misc.py:46: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


'1.7.6'

## Fractional charge molecules

In [2]:
from FcMole import FcM, FcM_like

In [3]:
mol_NN=gto.M(atom= "N 0 0 0; N 0 0 2.1",unit="Bohr", basis="sto-3g") 

The FcM_like function creates a copy of a givem PySCF molecule object with modified nuclear cherges 

In [4]:
fmol=FcM_like(mol_NN,fcs=[.2,-.1])

Only nuclear charges are modified, the number of electrons stays the same

In [5]:
fmol.atom_charges(),fmol.nelec

(array([7.2, 6.9]), (7, 7))

The same result can be achieved using the FcM constructor

In [6]:
fmol1=FcM(fcs=[.2,-.1],atom= "N 0 0 0; N 0 0 2.1",unit="Bohr", basis="sto-3g")

The number of electron matches the atom string

In [7]:
fmol1.atom_charges(),fmol1.nelec

(array([7.2, 6.9]), (7, 7))

If only some atoms have modified nuclear charges is possible to pass as a fcs parameter a double list [[idxs],[fcs]]

In [8]:
fmol2=FcM(fcs=[[0,1],[.2,-.1]],atom= "N 0 0 0; N 0 0 2.1",unit="Bohr", basis="sto-3g")

In [9]:
fmol2.atom_charges(),fmol2.nelec

(array([7.2, 6.9]), (7, 7))

The RKS-DFT objects are set.

In [10]:
mf=scf.RKS(fmol)
mf1=scf.RKS(fmol1)
mf2=scf.RKS(fmol2)

The KS-DFT functional is specified. Here, the PBE0 functional is used.

In [11]:
mf.xc = 'pbe0'
mf1.xc = 'pbe0'
mf2.xc = 'pbe0'

For SCF calculations using fractional charges molecules the initial density matrix guess needs to be evaluated from the eigenfunction of the monoelectronic Hamiltonian (there is no single atom density "SAD" matrix guess for atoms with fractional charge), i.e., init_guess_by_atom cannot be used.

In [12]:
mf.scf(dm0=mf.init_guess_by_1e())
mf1.scf(dm0=mf1.init_guess_by_1e())
mf2.scf(dm0=mf2.init_guess_by_1e())

converged SCF energy = -109.772234130647


/usr/local/lib/python3.9/site-packages/pyscf/gto/mole.py:1089: UserWarning: Function mol.dumps drops attribute with_rinv_at_nucleus because it is not JSON-serializable
  warnings.warn(msg)


converged SCF energy = -109.772234130647
converged SCF energy = -109.772234130647


-109.77223413064688

The standard KS-DFT (PBE0) calculations without modulating the nuclear charges.

In [13]:
mol_raw = gto.M(atom= "N 0 0 0; N 0 0 2.1",unit="Bohr", basis="sto-3g")
mf_raw = scf.RKS(mol_raw)
mf_raw.xc = 'pbe0'
mf_raw.scf(dm0=mf.init_guess_by_1e())

converged SCF energy = -107.9439147857


-107.9439147856995

In [14]:
%load_ext autoreload
%autoreload 2
from AP_class import APDFT_perturbator as AP

The alchemical perturbator is instantiated from a converged RKS object of the reference molecule, and for some given perturbation sites. In contrast to the above fractional charge calculation, the alchemical perturbator uses the SCF results without modulating the nuclear charges.

In [15]:
mf_nn=scf.RKS(mol_NN)
mf_nn.scf()
ap_nn=AP(mf_nn,sites=[0,1])

converged SCF energy = -107.154704428103


Alchemical gradient $ \partial E/\partial Z_i $, hessian $\partial^2E/\partial Z_i\partial Z_j$ and cubic hessian $\partial^3E/\partial Z_i\partial Z_j\partial Z_k$ can be obtained from their buid functions 


Electronic + nuclear alchemical gradient

In [16]:
#build the alchemical gradient dE/dZ_i
ap_nn.build_gradient()

array([-17.96610559, -17.96610559])

Electronic and nuclear alchemical gradients

In [17]:
ap_nn.build_elec_gradient()

array([-21.29943893, -21.29943893])

In [18]:
ap_nn.build_nuc_gradient()

array([3.33333333, 3.33333333])

Electronic + nuclear alchemical Hessian

In [19]:
#build the alchemical hessian d**2E/dZ_i/dZ_j
ap_nn.build_hessian()

array([[-0.53069508,  0.89997055],
       [ 0.89997055, -0.53069508]])

Electronic and nuclear alchemical Hessian

In [20]:
ap_nn.build_elec_hessian()

array([[-0.53069508,  0.42378008],
       [ 0.42378008, -0.53069508]])

In [21]:
ap_nn.build_nuc_hessian()

array([[0.        , 0.47619048],
       [0.47619048, 0.        ]])

In [ ]:
ap_nn.build_cubic_hessian()

Are saved inside the class and can be accessed in a later moment

In [ ]:
ap_nn.gradient,ap_nn.hessian,ap_nn.cubic_hessian

Alchemical perturbations for isoelectronic transmutations can be calculated from the derivatives up to order 3

In [ ]:
ap_nn.APDFT1(np.asarray([-1,0])) # to CN-

In [ ]:
ap_nn.APDFT2(np.asarray([-1,1])) # to CO

In [ ]:
ap_nn.APDFT3(np.asarray([0,1]))  # to NO+

In [ ]:
ap_nn.APDFT3(np.asarray([1,1])) # to OO++

## Alchemical forces
Alchemical forces are calculated resusing the density matrix derivatives already evaluated. The function af(i) gives the alchemical force of the atom $i$ : $\partial \mathbf{g}/ \partial Z_i$ [2]


In [ ]:
ap_nn.af(0),ap_nn.af(1)

## Basis set effects 
The class also include methods to calculate the energy of the target molecules, with its basis set and with the basis set of the reference,

In [ ]:
ap_nn.target_energy([-1,1]),ap_nn.target_energy_ref_bs([-1,1])

The APDFT3 predictions approximate the energy of the molecule with the reference basis set 

In [ ]:
ap_nn.APDFT3([-1,1])

But we can correct it using the single atom basis set correction [1]

In [ ]:
ap_nn.APDFT3([-1,1])+ap_nn.ap_bsec([-1,1])

## References 

[1] Giorgio Domenichini, Guido Falk von Rudorff, and O. Anatole von Lilienfeld : "Effects of perturbation order and basis set on alchemical predictions", J. Chem. Phys. 153, 144118 (2020)

[2] Giorgio Domenichini, and O. Anatole von Lilienfeld: "Alchemical predictions of relaxed geometries throughout chemical space", under review (2021) 